# Investor Lens Analyzer — Phase 1: Investor DNA

**To switch investors:** edit only the `⚙️ CONFIGURATION` cell — nothing else needs changing.

In [43]:
!pip install -q pypdf requests google-genai

In [44]:
import os
import re
import json
import time
import textwrap
import requests
import pandas as pd
from pathlib import Path
from typing import List, Dict
from pypdf import PdfReader

## ⚙️ CONFIGURATION — Edit this cell to switch investors

In [45]:
# ============================================================
#  USER CONFIG — only edit this cell to switch investors
# ============================================================

# Seth Klarman's link has 403 forbidden error, it can be replaced with other workable links later
INVESTOR_REGISTRY = {
    "warren_buffett": {
        "display_name": "Warren Buffett",
        "style": "Quality Compounder",
        "pdf_url": "https://12mv2.com/wp-content/uploads/2023/05/web_v31.pdf",
        "keyword_hints": ["moat", "float", "intrinsic value", "wonderful business", "Berkshire"]
    },
    "howard_marks": {
        "display_name": "Howard Marks",
        "style": "Credit/Cycles",
        "pdf_url": "https://www.oaktreecapital.com/docs/default-source/memos/the-complete-collection.pdf",
        "keyword_hints": ["risk", "cycle", "second-level", "uncertainty", "Oaktree"]
    },
    "seth_klarman": {
        "display_name": "Seth Klarman",
        "style": "Value/Distressed",
        "pdf_url": "https://www.safalniveshak.com/wp-content/uploads/2012/09/Seth-Klarman-Baupost-Group-Letters.pdf",
        "keyword_hints": ["margin of safety", "intrinsic value", "downside", "catalyst", "Baupost"]
    },
    "peter_lynch": {
        "display_name": "Peter Lynch",
        "style": "Value/Growth",
        "pdf_url": "https://12mv2.com/wp-content/uploads/2020/09/pl_fortworthcollection.pdf",
        "keyword_hints": ["tenbagger", "growth", "P/E", "story", "Fidelity", "Magellan"]
    },
    "nick_sleep": {
        "display_name": "Nick Sleep",
        "style": "Value/Distressed",
        "pdf_url": "https://igyfoundation.org.uk/wp-content/uploads/2021/03/Full_Collection_Nomad_Letters_.pdf",
        "keyword_hints": ["scale economics", "Nomad", "Amazon", "Costco", "destination"]
    },
}

# ← CHANGE THIS to switch investors
ACTIVE_INVESTOR = "warren_buffett"

# ============================================================
#  Derive all variables automatically — do not edit below
# ============================================================
cfg = INVESTOR_REGISTRY[ACTIVE_INVESTOR]
INVESTOR_NAME         = ACTIVE_INVESTOR
INVESTOR_DISPLAY_NAME = cfg["display_name"]
INVESTMENT_STYLE      = cfg["style"]
PDF_URL               = cfg["pdf_url"]
KEYWORD_HINTS         = cfg["keyword_hints"]
PDF_PATH              = Path(f"{INVESTOR_NAME}_letters.pdf")

print(f"✅ Investor : {INVESTOR_DISPLAY_NAME}")
print(f"   Style    : {INVESTMENT_STYLE}")
print(f"   PDF URL  : {PDF_URL}")

✅ Investor : Warren Buffett
   Style    : Quality Compounder
   PDF URL  : https://12mv2.com/wp-content/uploads/2023/05/web_v31.pdf


In [46]:
def download_pdf(url: str, out_path: Path):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f"Saved to {out_path} | size = {out_path.stat().st_size/1e6:.2f} MB")

download_pdf(PDF_URL, PDF_PATH)

Saved to warren_buffett_letters.pdf | size = 42.08 MB


In [47]:
def extract_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for i, page in enumerate(reader.pages):
        try:
            txt = page.extract_text()
        except Exception as e:
            print(f"Page {i} failed: {e}")
            txt = ""
        if txt:
            pages.append(txt)
    full_text = "\n\n".join(pages)
    return full_text

raw_text = extract_pdf_text(PDF_PATH)
print("Characters extracted:", len(raw_text))
print(raw_text[:2000])

Characters extracted: 12179678
More Compilations 
Warren: Buffett 
Through the Years 
B Y  W A R R E N  B U F F E T T
(Compilation, Introduction, and Categorization by Kevin G) 
If you have any Warren Buffett letters/transcripts/articles you’d like added to this compilation, please 
email me at kevin@12mv2.com. Please use the subject “Warren Buffett Compilation.” 


More Compilations 
 
 
 
 
 
 
 
Quote from Warren (Hopefully) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

More Compilations 
 
About Warren(In Progress, Kind of) 
Warren Buffett is the Chairman and CEO of Berkshire Hathaway. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


More Compilations 
 
Introduction(In Progress, Kind of) 
 
Warren is considered by many to be the greatest investor who ever lived.  

More Compilations 
About Warren Buffett 
Introduction by Kevin G (Work in Progress) 
Contents 
Buffett Partnership Letters 
1957-1970 
Unpublished Berkshire Hathaway Letters 
1969-1976 
Berkshire Hathaway Letters 
1977-2022
Berkshire Hat

In [48]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

clean = clean_text(raw_text)
print("Cleaned characters:", len(clean))
print(clean[:2000])

Cleaned characters: 12085910
More Compilations 
Warren: Buffett 
Through the Years 
B Y W A R R E N B U F F E T T
(Compilation, Introduction, and Categorization by Kevin G) 
If you have any Warren Buffett letters/transcripts/articles you’d like added to this compilation, please 
email me at kevin@12mv2.com. Please use the subject “Warren Buffett Compilation.” 

More Compilations 
 
 
 
 
 
 
 
Quote from Warren (Hopefully) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

More Compilations 
 
About Warren(In Progress, Kind of) 
Warren Buffett is the Chairman and CEO of Berkshire Hathaway. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

More Compilations 
 
Introduction(In Progress, Kind of) 
 
Warren is considered by many to be the greatest investor who ever lived. 

More Compilations 
About Warren Buffett 
Introduction by Kevin G (Work in Progress) 
Contents 
Buffett Partnership Letters 
1957-1970 
Unpublished Berkshire Hathaway Letters 
1969-1976 
Berkshire Hathaway Letters 
1977-2022
Berkshire Hathaway A

In [49]:
def chunk_text(text: str, max_chars: int = 7_000, overlap: int = 500) -> List[str]:
    # Create paragraph-based chunks safely.
    paragraphs = text.split("\n\n")
    chunks = []
    current = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        if len(current) + len(para) + 2 <= max_chars:
            current += para + "\n\n"
        else:
            if current.strip():
                chunks.append(current.strip())
            tail = current[-overlap:] if overlap > 0 else ""
            current = tail + "\n\n" + para + "\n\n"

    if current.strip():
        chunks.append(current.strip())

    return chunks

def score_chunk(chunk: str, keywords: List[str]) -> int:
    text_lower = chunk.lower()
    return sum(text_lower.count(k.lower()) for k in keywords)

TOP_N_CHUNKS = 20
chunks = chunk_text(clean, max_chars=7_000, overlap=500)
scored_chunks = [(i, ch, score_chunk(ch, KEYWORD_HINTS)) for i, ch in enumerate(chunks)]
scored_chunks = sorted(scored_chunks, key=lambda x: x[2], reverse=True)
top_chunks = [ch for _, ch, _ in scored_chunks[:TOP_N_CHUNKS]]

print(f"Total chunks created: {len(chunks)}")
print(f"Selected top chunks for Gemini: {len(top_chunks)}")
print("Top chunk scores:", [score for _, _, score in scored_chunks[:10]])



Total chunks created: 2483
Selected top chunks for Gemini: 20
Top chunk scores: [35, 30, 28, 28, 28, 27, 26, 26, 25, 24]


In [50]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request
from google import genai
from google.genai import types

PROJECT_ID = "analytics-in-practice-492209"
LOCATION   = "us-central1"
MODEL_NAME = "gemini-2.5-pro"

SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=SCOPES)
creds.refresh(Request())

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=creds
)

In [51]:
keyword_str = ", ".join(KEYWORD_HINTS)

EXTRACTION_PROMPT = f"""
You are extracting structured investment evidence from the writings of {INVESTOR_DISPLAY_NAME}.

Focus on capturing evidence related to these themes (specific to this investor):
{keyword_str}

From the provided text, extract evidence into these exact JSON keys:

{{
  "investment_philosophy": [],
  "invariable_truths": [],
  "buy_criteria": [],
  "avoid_red_flags": [],
  "position_sizing_portfolio": [],
  "sell_discipline": [],
  "risk_management": [],
  "signature_patterns": [],
  "blind_spots_mistakes": [],
  "notable_quotes": []
}}

Rules:
- Each item in a list should be a self-contained string (1-3 sentences).
- Prefer direct quotes for notable_quotes; paraphrase otherwise.
- Omit sections if no relevant evidence is found in this chunk.
- Return ONLY valid JSON — no markdown, no preamble.
"""

def extract_evidence_from_chunk(chunk: str) -> Dict:
    prompt = EXTRACTION_PROMPT + "\n\nTEXT:\n" + chunk
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1, max_output_tokens=4096)
    )
    raw = response.text.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {}

all_evidence = []
for i, chunk in enumerate(top_chunks):
    print(f"Processing chunk {i+1}/{len(top_chunks)}...")
    ev = extract_evidence_from_chunk(chunk)
    all_evidence.append(ev)
    time.sleep(1)

print("\nExtraction complete")


Processing chunk 1/20...
Processing chunk 2/20...
Processing chunk 3/20...
Processing chunk 4/20...
Processing chunk 5/20...
Processing chunk 6/20...
Processing chunk 7/20...
Processing chunk 8/20...
Processing chunk 9/20...
Processing chunk 10/20...
Processing chunk 11/20...
Processing chunk 12/20...
Processing chunk 13/20...
Processing chunk 14/20...
Processing chunk 15/20...
Processing chunk 16/20...
Processing chunk 17/20...
Processing chunk 18/20...
Processing chunk 19/20...
Processing chunk 20/20...

Extraction complete


In [52]:
SECTIONS = [
    "investment_philosophy", "invariable_truths", "buy_criteria", "avoid_red_flags",
    "position_sizing_portfolio", "sell_discipline", "risk_management",
    "signature_patterns", "blind_spots_mistakes", "notable_quotes"
]

def merge_evidence(chunks_evidence: List[Dict]) -> Dict:
    merged = {k: [] for k in SECTIONS}
    for chunk_ev in chunks_evidence:
        for section in SECTIONS:
            items = chunk_ev.get(section, [])
            if isinstance(items, list):
                merged[section].extend(items)
    return merged

def deduplicate(evidence: Dict) -> Dict:
    deduped = {}
    for section, items in evidence.items():
        seen = []
        for item in items:
            item_lower = item.lower()
            if not any(item_lower in s.lower() or s.lower() in item_lower for s in seen):
                seen.append(item)
        deduped[section] = seen
    return deduped

merged_evidence = deduplicate(merge_evidence(all_evidence))

for k, v in merged_evidence.items():
    print(f"  {k}: {len(v)} items")

  investment_philosophy: 67 items
  invariable_truths: 36 items
  buy_criteria: 22 items
  avoid_red_flags: 15 items
  position_sizing_portfolio: 7 items
  sell_discipline: 3 items
  risk_management: 24 items
  signature_patterns: 50 items
  blind_spots_mistakes: 11 items
  notable_quotes: 73 items


In [53]:
FINAL_DNA_PROMPT = f"""
You are creating a polished Investor DNA document for {INVESTOR_DISPLAY_NAME}.

Use the structured evidence provided below to produce a final output in this exact section structure:

INVESTOR DNA: {INVESTOR_DISPLAY_NAME}

1. INVESTMENT PHILOSOPHY
2. INVARIABLE TRUTHS (Market Axioms)
3. WHAT THEY LOOK FOR (Buy Criteria)
4. WHAT THEY AVOID (Red Flags)
5. POSITION SIZING & PORTFOLIO
6. SELL DISCIPLINE
7. RISK MANAGEMENT
8. SIGNATURE PATTERNS
9. KNOWN BLIND SPOTS & MISTAKES
10. NOTABLE QUOTES
11. HISTORICAL EXAMPLES

Rules:
- Write clearly and specifically.
- Do not invent unsupported details.
- If something is a reasonable inference rather than directly stated, label it as "Inference:".
- Keep the tone professional and analytical.
- Include 5-10 invariable truths if enough support exists.
- Include representative quotes.
- Output in clean markdown.
"""

def build_final_dna(evidence: Dict) -> str:
    payload = json.dumps(evidence, indent=2, ensure_ascii=False)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=FINAL_DNA_PROMPT + "\n\nEVIDENCE:\n" + payload
    )
    return response.text

final_dna_markdown = build_final_dna(merged_evidence)
print(final_dna_markdown)

**INVESTOR DNA: Warren Buffett**

### 1. INVESTMENT PHILOSOPHY

*   **Primacy of Intrinsic Value:** The ultimate goal is to increase the intrinsic business value of the company's shares. Intrinsic value is defined as the discounted value of the cash that can be taken out of a business during its remaining life. While impossible to pinpoint precisely, it is the only logical metric for evaluating investments.
*   **Focus on Per-Share Growth:** The primary objective is to increase per-share intrinsic value, not just aggregate earnings. This instills a discipline against dilutive acquisitions or actions that grow the empire without benefiting existing owners.
*   **Business-Picker, Not Stock-Picker:** Investments are made based on the long-term performance expectations of the underlying business. Stocks are viewed as pieces of a business, not as tickers for timely market moves. The goal is to own meaningful stakes in businesses with durable economic advantages and first-class management.
*

In [54]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
OUTPUT_DIR = Path("/content/drive/MyDrive/UC-A8 Investor Lens Analyzer/Investor DNA")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save Markdown
(OUTPUT_DIR / f"{INVESTOR_NAME}_dna.md").write_text(
    final_dna_markdown,
    encoding="utf-8"
)

# Save JSON
(OUTPUT_DIR / f"{INVESTOR_NAME}_evidence.json").write_text(
    json.dumps(merged_evidence, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Save CSV
rows = []
for section, items in merged_evidence.items():
    for item in items:
        rows.append({
            "investor": INVESTOR_NAME,
            "section": section,
            "content": item
        })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / f"{INVESTOR_NAME}_evidence.csv", index=False)

print("Saved to Google Drive:")
print(OUTPUT_DIR)

Saved to Google Drive:
/content/drive/MyDrive/UC-A8 Investor Lens Analyzer/Investor DNA
